<a href="https://colab.research.google.com/github/brugerard/Test/blob/main/Transferability_FastUpload_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Transferability Potential — Fast Upload-First Colab Notebook
This version **avoids slow Drive scans**. It opens a file picker immediately (Excel/CSV), with an optional manual Drive path override if you prefer to load from Drive. Earth Engine uses your project **`java-brugerard`**.


## 0) Setup

In [1]:

# If needed on a fresh runtime:
# %pip install -q earthengine-api geemap pandas openpyxl

import os, re, json
import numpy as np
import pandas as pd

from google.colab import drive, files
drive.mount('/content/drive')

import ee, geemap
EE_PROJECT = "java-brugerard"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize(project=EE_PROJECT)

try:
    geemap.ee_initialize(project=EE_PROJECT)
except Exception:
    pass

print("✓ Earth Engine ready with project:", EE_PROJECT)


Mounted at /content/drive
To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=DzdjWHf4TWQcXgMGdatSsYohFGecMdEwwtg3usfAcGU&tc=vxKHvdRkxISdcUmsZ92uAhs1amVXHPiT-pAGYbc3YHQ&cc=d5AuB9DELPF_sY0orhYw0KvCtGHp4uhKnmhVvy62dcs

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1Ab32j90Pi7Ko3JoqmMBYg8pcZvlHxnICpzTzRNKDKQ4cB3mNu9GJin6RXcE

Successfully saved authorization token.
✓ Earth Engine ready with project: java-brugerard


## 1) Configuration

In [2]:

CONFIG = {
    "country": "Zimbabwe",
    "admin_fc": "USDOS/LSIB_SIMPLE/2017",
    "admin_country_prop": "country_na",
    "worldcover": {"id": "ESA/WorldCover/v200", "year": 2021, "band": "Map", "cropland_class": 40},
    "elevation": {"id": "USGS/SRTMGL1_003", "band": "elevation"},
    "worldclim_tavg": {"id": "WORLDCLIM/V1/MONTHLY", "band_prefix": "tavg_"},
    "worldclim_prec": {"id": "WORLDCLIM/V1/MONTHLY", "band_prefix": "prec_"},
    "soilgrids": {"id": "ISRIC/SoilGrids/250m",
                  "bands": ["sand_0-5cm_mean", "sand_5-15cm_mean", "sand_15-30cm_mean"],
                  "weights": [5, 10, 15]},
    "pop": {"id": "CIESIN/GPWv411/GPW_UNWPP-Adjusted_Population_Density", "band": "unwpp-adjusted_population_density_2020"},
    "access": {"id": "Oxford/MAP/accessibility_to_cities_2015_v1.0", "band": "accessibility"},
}
CRS = "EPSG:4326"
SCALE = 1000  # meters


## 2) Choose your table (upload now) or set a Drive path manually

In [3]:

from google.colab import files
import pandas as pd, os, re

# Preferred: upload a local Excel/CSV (fast, reliable)
print("Choose your Excel/CSV to upload…")
up = files.upload()  # opens file picker
assert len(up) == 1, "Upload exactly one file."
table_path = f"/content/{list(up.keys())[0]}"
print("✓ Uploaded:", table_path)

# OPTIONAL: If you prefer to read from Drive instead, set DRIVE_PATH and uncomment:
# DRIVE_PATH = "/content/drive/MyDrive/YourFolder/BASELINE DATA.xlsx"
# if os.path.exists(DRIVE_PATH):
#     table_path = DRIVE_PATH
#     print("✓ Using Drive file:", table_path)

# Read Excel first, fallback to CSV
try:
    hh = pd.read_excel(table_path)
except Exception:
    hh = pd.read_csv(table_path)

print("Columns detected:", list(hh.columns))

# Auto-detect lon/lat; allow override
lon_pat = re.compile(r"(lon|long|longitude|x_coord|^x$)", re.I)
lat_pat = re.compile(r"(lat|latitude|y_coord|^y$)", re.I)
lon_col = next((c for c in hh.columns if lon_pat.search(str(c))), None)
lat_col = next((c for c in hh.columns if lat_pat.search(str(c))), None)
print(f"Auto-detected longitude column: {lon_col}")
print(f"Auto-detected latitude  column: {lat_col}")

ovr = input("Press Enter to accept, or type 'edit' to override lon/lat: ").strip().lower()
if ovr == "edit":
    lon_col = input("Longitude column name: ").strip()
    lat_col = input("Latitude column name: ").strip()

for col in (lon_col, lat_col):
    if col not in hh.columns:
        raise KeyError(f"Column '{col}' not in table. Available: {list(hh.columns)}")

hh_xy = hh[[lon_col, lat_col]].dropna()
print(f"✓ Points loaded: {len(hh_xy)}")


Choose your Excel/CSV to upload…


Saving BASELINE DATA.xlsx to BASELINE DATA.xlsx
✓ Uploaded: /content/BASELINE DATA.xlsx
Columns detected: ['start', 'end', 'Has this farm already been interviewed earlier or is this a new farm?', 'District', 'Ward', 'District.1', 'Ward.1', 'Enter a date and time', 'Record your current location', '_Record your current location_latitude', '_Record your current location_longitude', '_Record your current location_altitude', '_Record your current location_precision', 'How old is the head of the household?', 'What is the sex of the head of the household?', 'What is the education level of the head of the household?', 'Total number of adult males aged 61 or more', 'Total number of adult females aged 61 or more', 'Total number of adult males aged 25-60', 'Total number of adult females aged 25-60', 'Total number of young males, aged 15-25', 'Total number of young females, aged 15-25', 'Total number of children and teens, aged 3-14', 'Total number of infants of age 0 to 2', 'Nb of tractors?', 'Nb

## 3) Helper functions

In [4]:

def get_country_geom(country_name: str) -> ee.Geometry:
    fc = ee.FeatureCollection(CONFIG["admin_fc"])
    props = ["country_na", "country_na_2", "name", "Country"]
    filt = None
    for p in props:
        try:
            sub = fc.filter(ee.Filter.eq(p, country_name))
            if sub.size().getInfo() > 0:
                filt = sub
                break
        except Exception:
            pass
    if filt is None:
        raise ValueError("Country not found in admin FC. Adjust CONFIG['admin_fc'] or property name.")
    return filt.geometry()

def zscore_stack(img: ee.Image, region: ee.Geometry, scale: int) -> ee.Image:
    stats = img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=region, scale=scale, maxPixels=1e13, bestEffort=True
    )
    z_imgs = []
    for b in img.bandNames().getInfo():
        mu = ee.Number(stats.get(b + "_mean"))
        sd = ee.Number(stats.get(b + "_stdDev"))
        z = img.select(b).subtract(mu).divide(sd).rename(b + "_z")
        z_imgs.append(z)
    return ee.Image.cat(z_imgs)

def euclidean_distance(img: ee.Image, center: dict) -> ee.Image:
    diffs = []
    for b in img.bandNames().getInfo():
        mu = ee.Number(center.get(b, 0))
        d = img.select(b).subtract(mu).pow(2)
        diffs.append(d)
    return ee.ImageCollection(diffs).sum().sqrt().rename("dist")

def tp_from_distance(dist: ee.Image) -> ee.Image:
    return dist.add(1).pow(-1).rename("tp")


## 4) Build feature stack

In [11]:
# --- 4) Build feature stack (bulletproof fallbacks + clear logging) ---
region = get_country_geom(CONFIG["country"])

def _ic_first_safe(ic, **kwargs):
    """Return first image of a collection or None if empty/inaccessible."""
    try:
        col = ee.ImageCollection(ic)
        for k,v in kwargs.items():
            if k == "filter":
                col = col.filter(v)
        img = col.first()
        # force server check
        _ = img.bandNames().getInfo()
        return img
    except Exception:
        return None

def _img_safe(id_str):
    """Return ee.Image or None."""
    try:
        img = ee.Image(id_str)
        _ = img.bandNames().getInfo()
        return img
    except Exception:
        return None

# --- Cropland mask (try ESA WorldCover, then MODIS MCD12Q1, else no mask) ---
cropland = None
wc = None
try:
    wc = ee.ImageCollection("ESA/WorldCover/v200").filter(ee.Filter.eq("year", 2021)).first()
    _ = wc.bandNames().getInfo()
    wc_map = wc.select("Map")
    cropland = wc_map.eq(40).selfMask()  # 40=cropland
    print("✓ Cropland: ESA WorldCover v200 (2021, class 40)")
except Exception as e:
    print("⚠️ ESA WorldCover 2021 not available:", str(e))
    # Fallback: MODIS MCD12Q1 IGBP classes (12=croplands, 14=cropland/natural mosaic)
    mcd = _img_safe("MODIS/006/MCD12Q1/2019_01_01")
    if mcd:
        igbp = mcd.select("LC_Type1")
        cropland = igbp.eq(12).Or(igbp.eq(14)).selfMask()
        print("✓ Cropland: MODIS MCD12Q1 (IGBP classes 12,14)")
    else:
        print("⚠️ No cropland mask available; proceeding WITHOUT cropland masking.")

# --- Elevation (SRTM) ---
elev = _img_safe("USGS/SRTMGL1_003")
if not elev:
    raise RuntimeError("SRTM elevation unavailable; cannot proceed.")
elev = elev.select("elevation").rename("elevation")
print("✓ Elevation: SRTM")

# --- Climate (WorldClim monthly → annual; fallback BIO) ---
tavg, prec = None, None
wc_monthly = _img_safe("WORLDCLIM/V1/MONTHLY")
if wc_monthly:
    try:
        tavg = ee.Image.cat([wc_monthly.select(f"tavg_{i:02d}") for i in range(1,13)]).reduce(ee.Reducer.mean()).rename("tavg")
        prec = ee.Image.cat([wc_monthly.select(f"prec_{i:02d}") for i in range(1,13)]).reduce(ee.Reducer.sum()).rename("prec")
        print("✓ Climate: WorldClim V1 monthly (tavg mean, prec sum)")
    except Exception as e:
        print("⚠️ WorldClim monthly bands not accessible:", str(e))
if tavg is None or prec is None:
    bio = _img_safe("WORLDCLIM/V1/BIO")
    if not bio:
        raise RuntimeError("WorldClim climate unavailable; cannot proceed.")
    tavg = bio.select("bio01").divide(10).rename("tavg")  # BIO1*10
    prec = bio.select("bio12").rename("prec")
    print("✓ Climate: WorldClim BIO (bio01→tavg, bio12→prec)")

# --- Soil sand 0–30 cm (SoilGrids if available; else skip) ---
sand = None
sg = _img_safe("projects/soilgrids-isric/soilgrids")
if sg:
    try:
        names = sg.bandNames().getInfo()
        def _pick(bands, *needles):
            needles = [n.lower() for n in needles]
            for b in bands:
                bl = b.lower()
                if all(n in bl for n in needles):
                    return b
            return None
        b05   = _pick(names, "sand", "0",  "5",  "mean")
        b515  = _pick(names, "sand", "5",  "15", "mean")
        b1530 = _pick(names, "sand", "15", "30", "mean")
        if b05 and b515 and b1530:
            w = CONFIG["soilgrids"]["weights"]
            sand = (sg.select(b05).multiply(w[0])
                    .add(sg.select(b515).multiply(w[1]))
                    .add(sg.select(b1530).multiply(w[2]))
                    .divide(sum(w))
                    .rename("sand"))
            print("✓ Soil: SoilGrids sand (weighted 0–30 cm)")
        else:
            print("⚠️ SoilGrids present but expected sand bands not found; skipping sand.")
    except Exception as e:
        print("⚠️ SoilGrids error; skipping sand:", str(e))
else:
    print("⚠️ Soil: SoilGrids not accessible; skipping sand.")

# --- Population (try GPW unadjusted → GPW adjusted → WorldPop mosaic) ---
pop = None
gpw_unadj = _ic_first_safe("CIESIN/GPWv411/GPW_Population_Density", filter=ee.Filter.eq('year', 2020))
if gpw_unadj:
    try:
        pop = gpw_unadj.select('population_density').rename('pop')
        print("✓ Population: GPWv4 Population Density (2020)")
    except Exception as e:
        print("⚠️ GPWv4 unadjusted select failed:", str(e))

if pop is None:
    gpw_adj = _ic_first_safe("CIESIN/GPWv411/GPW_UNWPP-Adjusted_Population_Density", filter=ee.Filter.eq('year', 2020))
    if gpw_adj:
        try:
            pop = gpw_adj.select('unwpp-adjusted_population_density').rename('pop')
            print("✓ Population: GPWv4 UNWPP-adjusted (2020)")
        except Exception as e:
            print("⚠️ GPWv4 adjusted select failed:", str(e))

if pop is None:
    # WorldPop filtered by region & date; mosaic in case multiple tiles
    wp = ee.ImageCollection("WorldPop/GP/100m/pop").filterBounds(region).filterDate('2020-01-01','2021-01-01')
    try:
        # Some regions may have no tiles; if so, try global 2020 mosaic
        wp_img = wp.select('population').mosaic()
        _ = wp_img.bandNames().getInfo()  # check
        pop = wp_img.rename('pop')
        print("✓ Population: WorldPop 2020 (mosaicked, region-filtered)")
    except Exception as e:
        print("⚠️ WorldPop region-filtered missing; trying global mosaic:", str(e))
        wp2 = ee.ImageCollection("WorldPop/GP/100m/pop").filterDate('2020-01-01','2021-01-01').select('population').mosaic()
        _ = wp2.bandNames().getInfo()
        pop = wp2.rename('pop')
        print("✓ Population: WorldPop 2020 (mosaicked, global)")

# --- Accessibility (Oxford → VIIRS proxy) ---
access = _img_safe("Oxford/MAP/accessibility_to_cities_2015_v1.0")
if access:
    try:
        access = access.select("accessibility").rename("travel_time")
        print("✓ Accessibility: Oxford/MAP (2015)")
    except Exception as e:
        print("⚠️ Oxford select failed; using VIIRS proxy:", str(e))
        access = None
if access is None:
    viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG") \
              .filterDate("2020-01-01", "2021-01-01") \
              .select("avg_rad") \
              .mean() \
              .rename("travel_time")
    access = viirs
    print("✓ Accessibility proxy: VIIRS nightlights 2020 mean")

def prep(img):
    img = img.clip(region)
    if cropland:
        img = img.updateMask(cropland)
    return img

# Build final stack; include sand only if available
stack_bands = [prep(elev)]
if sand is not None:
    stack_bands.append(prep(sand))
stack_bands += [prep(tavg), prep(prec), prep(pop), prep(access)]

stack = ee.Image.cat(stack_bands).reproject(crs=CRS, scale=SCALE).setDefaultProjection(CRS, None, SCALE)
print("Stack bands:", stack.bandNames().getInfo())


⚠️ ESA WorldCover 2021 not available: Image.bandNames: Parameter 'image' is required and may not be null.
✓ Cropland: MODIS MCD12Q1 (IGBP classes 12,14)
✓ Elevation: SRTM
✓ Climate: WorldClim BIO (bio01→tavg, bio12→prec)
⚠️ Soil: SoilGrids not accessible; skipping sand.
✓ Population: WorldPop 2020 (mosaicked, region-filtered)
✓ Accessibility proxy: VIIRS nightlights 2020 mean
Stack bands: ['elevation', 'tavg', 'prec', 'pop', 'travel_time']


## 5) Build archetype FeatureCollection

In [12]:

features = [
    ee.Feature(ee.Geometry.Point([float(r[lon_col]), float(r[lat_col])]), {})
    for _, r in hh_xy.iterrows()
]
fc_site = ee.FeatureCollection(features)
centroid = ee.Geometry.MultiPoint(fc_site.geometry().coordinates()).centroid().coordinates()
print("Archetype centroid [lon, lat]:", centroid.getInfo(), "| n points:", len(features))


Archetype centroid [lon, lat]: [30.44155796266663, -16.114381006396723] | n points: 270


## 6) Z-scores → Distance → TP → Quartiles

In [14]:
# --- 6) Z-scores → Distance → TP → Quartiles (robust sampling + safe distance math) ---

# 1) Z-score the stack (same region & scale you used earlier)
stack_z = zscore_stack(stack, region=region, scale=SCALE)

# 2) Try sampling AT the points, but on an UNMASKED z-image so cropland mask can't drop them
samples = stack_z.unmask().sampleRegions(
    collection=fc_site,
    scale=SCALE,
    geometries=False
).toList(100000)

n = samples.size().getInfo()
rows = [samples.get(i).getInfo()["properties"] for i in range(n)]

# 3) If still no values (n==0 or all None), buffer points (1 km) and take mean inside each buffer
if n == 0 or all(len([v for v in r.values() if v is not None]) == 0 for r in rows):
    print("⚠️ Point sampling returned no data (likely masked). Trying 1 km buffers with mean reducer…")
    fc_buffers = fc_site.map(lambda f: f.buffer(1000))  # 1000 m
    red = stack_z.reduceRegions(
        collection=fc_buffers,
        reducer=ee.Reducer.mean(),
        scale=SCALE
    ).toList(100000)
    n = red.size().getInfo()
    rows = [red.get(i).getInfo()["properties"] for i in range(n)]

# 4) Compute archetype mean per band from the sampled records
from collections import defaultdict
vals = defaultdict(list)
for r in rows:
    for k, v in r.items():
        if v is not None and isinstance(v, (int, float)):
            vals[k].append(v)

# Keep only z-bands actually present
bandnames = stack_z.bandNames().getInfo()
archetype_mean = {k: float(np.mean(v)) for k, v in vals.items() if k in bandnames and len(v) > 0}
print("Archetype z-means (preview):", {k: round(archetype_mean[k], 3) for k in list(archetype_mean.keys())[:4]}, "...")

# Guard: if still empty, bail with a clear message
if len(archetype_mean) == 0:
    raise RuntimeError("No valid archetype samples found. Check that your lon/lat fall inside Zimbabwe "
                       "and that masking hasn’t removed all pixels near your points (try larger buffer or skip cropland mask).")

# 5) Build a per-band constant “center image” from archetype means
def center_image_from_dict(band_list, center_dict):
    imgs = []
    for b in band_list:
        mu = center_dict.get(b, None)
        if mu is None:
            # If a band wasn't sampled, set center to 0 (z=0), i.e., it won’t affect distance
            mu = 0.0
        imgs.append(ee.Image.constant(mu).rename(b))
    return ee.Image.cat(imgs)

center_img = center_image_from_dict(bandnames, archetype_mean)

# 6) Distance: (stack_z - center)^2 summed across bands, then sqrt
diff_sq = stack_z.select(bandnames).subtract(center_img).pow(2)
# sum across bands for each pixel:
dist = diff_sq.reduce(ee.Reducer.sum()).sqrt().rename("dist")

# 7) Transferability potential & quantiles
tp = dist.add(1).pow(-1).rename("tp")

percentiles = tp.reduceRegion(
    reducer=ee.Reducer.percentile([0,25,50,75,100], maxBuckets=1e6),
    geometry=region, scale=SCALE, maxPixels=1e13, bestEffort=True
).getInfo()
q0, q25, q50, q75, q100 = [percentiles[f"tp_p{p}"] for p in [0,25,50,75,100]]
print("TP quantiles:", dict(q0=q0, q25=q25, q50=q50, q75=q75, q100=q100))

# Class 1..4
tp_q = (tp.gt(q25).add(tp.gt(q50)).add(tp.gt(q75)).add(tp.gt(q100))).max(1).rename("tp_class")


Archetype z-means (preview): {'elevation_z': 0.0, 'pop_z': 0.0, 'prec_z': 0.0, 'tavg_z': 0.0} ...
TP quantiles: {'q0': 0.03690836138080777, 'q25': 0.34429288966995725, 'q50': 0.4458784594347012, 'q75': 0.5469637941201112, 'q100': 0.7908391623780066}


## 7) Map & Export (optional)

In [15]:

m = geemap.Map(center=[centroid.get(1).getInfo(), centroid.get(0).getInfo()], zoom=6)
m.addLayer(dist, {"min": 0, "max": 6}, "Distance to archetype")
m.addLayer(tp, {"min": 0, "max": 1}, "Transferability potential")
m.addLayer(tp_q, {"min": 1, "max": 4, "palette": ["#fde725", "#5ec962", "#21918c", "#3b528b"]}, "TP classes")
m.addLayer(ee.FeatureCollection(CONFIG["admin_fc"]).filter(ee.Filter.eq(CONFIG["admin_country_prop"], CONFIG["country"])), {}, "Country")
m.addLayer(fc_site, {"color": "red"}, "Archetype points")
m

# Optional exports:
# geemap.ee_export_image(dist, filename='/content/drive/MyDrive/dist_to_archetype.tif', scale=SCALE, region=region, file_per_band=True)
# geemap.ee_export_image(tp, filename='/content/drive/MyDrive/transferability_potential.tif', scale=SCALE, region=region, file_per_band=True)
# geemap.ee_export_image(tp_q, filename='/content/drive/MyDrive/transferability_classes.tif', scale=SCALE, region=region, file_per_band=True)


Map(center=[-16.114381006396723, 30.44155796266663], controls=(WidgetControl(options=['position', 'transparent…